# 01 — Explore historical arrival/departure data

Exploration only: nothing here defines the pipeline (that belongs in `src/`).

**Source:** MBTA Bus Arrival & Departure Times 2026, read straight out of `data/raw/mbta_2026.zip` (Jan–Sep).
`data/raw/` is read-only, so this notebook never extracts or writes there.

**Label rule:** late = strictly more than 3 minutes (180 s) after scheduled time. Exactly 180 s is on time.

In [ ]:
from pathlib import Path
import zipfile

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ZIP_PATH = ROOT / "data" / "raw" / "mbta_2026.zip"
LATE_SECONDS = 180

# Each month is ~2.5M rows / ~270 MB. Add months here once memory allows (the zip holds 01–09).
MONTHS = ["07", "08", "09"]

## Files available

In [ ]:
with zipfile.ZipFile(ZIP_PATH) as zf:
    members = [i for i in zf.infolist() if i.filename.endswith(".csv")]
pd.DataFrame({"file": [Path(i.filename).name for i in members],
              "size_mb": [round(i.file_size / 1e6) for i in members]})

## Load

Low-cardinality text columns are read as `category` to keep memory down.

In [ ]:
DTYPES = {
    "route_id": "category", "direction_id": "category", "half_trip_id": "string",
    "stop_id": "category", "time_point_id": "category", "time_point_order": "Int16",
    "point_type": "category", "standard_type": "category",
    "scheduled_headway": "Int32", "headway": "Int32",
}


def read_month(month):
    name = f"MBTA_Bus_Arrival_Departure_Times_2026/MBTA-Bus-Arrival-Departure-Times_2026-{month}.csv"
    with zipfile.ZipFile(ZIP_PATH) as zf, zf.open(name) as f:
        return pd.read_csv(f, dtype=DTYPES, parse_dates=["service_date"])


df = pd.concat([read_month(m) for m in MONTHS], ignore_index=True)
for col in ["route_id", "direction_id", "stop_id", "time_point_id", "point_type", "standard_type"]:
    df[col] = df[col].astype("category")  # concat of differing categories falls back to object
print(f"{len(df):,} rows, {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
df.head()

In [ ]:
df.isna().mean().round(3).rename("null_share").to_frame()

## Times

`scheduled` and `actual` are clock times on a dummy 1900-01-01 date. Trips after midnight roll into 1900-01-02,
so they're offsets from the start of `service_date`, not real timestamps. The delay is just `actual - scheduled`.

In [ ]:
BASE = pd.Timestamp("1900-01-01")
sched_offset = pd.to_datetime(df["scheduled"]).dt.tz_localize(None) - BASE
actual_offset = pd.to_datetime(df["actual"]).dt.tz_localize(None) - BASE

df["scheduled_at"] = df["service_date"] + sched_offset
df["delay_s"] = (actual_offset - sched_offset).dt.total_seconds()
df["is_late"] = (df["delay_s"] > LATE_SECONDS).astype("boolean")  # strict >
df.loc[df["delay_s"].isna(), "is_late"] = pd.NA

print("share of rows past midnight:", round((sched_offset >= pd.Timedelta(days=1)).mean(), 3))
df["delay_s"].describe().round(0)

Hour-of-day volume should peak at the morning and evening rush. If it does, the `Z` suffix is misleading and the times are Boston local time.

In [ ]:
df["scheduled_at"].dt.hour.value_counts().sort_index().plot.bar(figsize=(10, 3), title="Rows by scheduled hour")
plt.show()

## Categorical columns

In [ ]:
for col in ["direction_id", "point_type", "standard_type"]:
    print(df[col].value_counts(), "\n")
print("routes:", df["route_id"].nunique(), "| stops:", df["stop_id"].nunique(), "| timepoints:", df["time_point_id"].nunique())

## Missing actuals

The spec drops departures with no recorded time instead of counting them as on time. Check where the missing values are.

In [ ]:
df.groupby(["point_type", "standard_type"], observed=True)["actual"].apply(lambda s: s.isna().mean()).unstack().round(3)

## Delay distribution and late rate

In [ ]:
observed = df.dropna(subset=["delay_s"]).copy()
observed["is_late"] = observed["is_late"].astype(bool)

ax = (observed["delay_s"].clip(-900, 1800) / 60).plot.hist(bins=135, figsize=(10, 3), title="Delay (minutes, clipped to -15..30)")
ax.axvline(LATE_SECONDS / 60, color="red", linestyle="--", label="3 min")
ax.legend()
plt.show()

print(f"late rate (strict > 180 s): {observed['is_late'].mean():.3f}")
print(f"rows at exactly 180 s (on time by rule): {(observed['delay_s'] == LATE_SECONDS).sum():,}")
print(f"late rate if 180 s were counted late:   {(observed['delay_s'] >= LATE_SECONDS).mean():.3f}")

### By `standard_type` and `point_type`

On **Headway** routes the agency judges service by spacing between buses, not by schedule adherence, so a schedule-based "late" may mean less there.
For **Endpoint** rows `actual` is an arrival; for Start/Midpoints it's a departure.

In [ ]:
observed.pivot_table(index="point_type", columns="standard_type", values="is_late", aggfunc="mean", observed=True).round(3)

### By hour, day of week, and route

In [ ]:
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
observed.groupby(observed["scheduled_at"].dt.hour)["is_late"].mean().plot.bar(ax=axes[0], title="Late rate by scheduled hour")
observed.groupby(observed["service_date"].dt.day_name())["is_late"].mean().reindex(DAYS).plot.bar(ax=axes[1], title="Late rate by day")
plt.tight_layout()
plt.show()

In [ ]:
by_route = observed.groupby("route_id", observed=True)["is_late"].agg(late_rate="mean", rows="size")
by_route = by_route[by_route["rows"] >= 5000].sort_values("late_rate")
pd.concat([by_route.head(10), by_route.tail(10)]).round(3)

### Over time

Look for schedule-change weeks, holidays, or gaps in the record.

In [ ]:
observed.groupby("service_date")["is_late"].mean().plot(figsize=(12, 3), title="Daily late rate")
plt.show()

## Leakage check

Which columns can be features? Only what's known **before** the trip runs.

| column | known before the trip? |
|---|---|
| `service_date`, `route_id`, `direction_id`, `stop_id`, `time_point_id`, `time_point_order`, `point_type`, `standard_type`, `scheduled`, `scheduled_headway` | yes, from the schedule |
| `actual`, `headway` | **no**, recorded during the trip. Label source only. |
| `half_trip_id` | trip identifier, use for joins, not as a feature |

Historical late rates (for example the last 8 weeks for this slot) are fine, as long as the window ends **before** the trip's service date.

## Notes to carry forward

- Late rate across all timepoints is well above the ~25% the spec assumed, so check it per `point_type` / `standard_type` before setting the target.
- The rule's strict `>` matters: there are thousands of rows at exactly 180 s.
- Decide how Headway-standard routes are labelled (open question in `ml_spec.md`).
- Missing `actual` values are dropped, per the spec.
- Only Jul–Sep are loaded here. The zip has Jan–Sep, and the spec wants at least 12 months.